<a href="https://colab.research.google.com/github/RifaDeen/symAD-ECNN/blob/main/notebooks/models/07_ecnn_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ E(n)-Equivariant CNN-Autoencoder for Brain MRI Anomaly Detection

## 📋 Overview

This notebook implements the **MAIN MODEL** - an **E(2)-Equivariant CNN Autoencoder** using the `e2cnn` library.

### Key Innovation: Rotation Equivariance
- **Standard CNN**: NOT rotation-invariant → high false positives on rotated tumors
- **ECNN (this model)**: Handles rotations **internally** → reduced false positives
- **Group Theory**: Uses C4 group (4 discrete rotations: 0°, 90°, 180°, 270°)

### Model Architecture
- **Type**: E(2)-Equivariant Convolutional Autoencoder
- **Library**: `e2cnn` (E(n)-Equivariant CNN)
- **Group**: C4 (4-fold rotational symmetry)
- **Layers**: R2Conv (rotation-equivariant convolutions)
- **Feature Type**: Regular representation of C4

### Why This Matters
- **Medical Imaging**: Tumors appear at arbitrary orientations
- **Without ECNN**: Need data augmentation (rotation) → slower training
- **With ECNN**: Built-in rotation handling → faster + more accurate

### Expected Performance
- **AUROC**: ~0.88-0.92 (**BEST** of all 3 models)
- **False Positive Rate**: ~30% lower than CNN-AE
- **Training Time**: ~40-50 minutes on Colab GPU

---

## 1️⃣ Setup and Environment

In [ ]:
# Mount Drive and install packages (INCLUDING e2cnn!)

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Install e2cnn for equivariant convolutions
!pip install e2cnn -q
!pip install pytorch-msssim -q

print("✅ e2cnn and other packages installed!")

In [ ]:
# ==========================================
# ⚡ TURBO LOADER (Unzip to Local)
# ==========================================
import zipfile
import os
import shutil

BASE_DIR = "/content/drive/MyDrive/symAD-ECNN/data"
LOCAL_DIR = "/content/local_data"

ZIPS = {
    "train": f"{BASE_DIR}/train_fast.zip",
    "val":   f"{BASE_DIR}/val_fast.zip",
    "test":  f"{BASE_DIR}/test_fast.zip"
}

print("🚀 Extracting to Local Disk...")

for name, zip_path in ZIPS.items():
    target_dir = f"{LOCAL_DIR}/{name}"
    # Clean up old run if exists
    if os.path.exists(target_dir): 
        shutil.rmtree(target_dir)
    os.makedirs(target_dir, exist_ok=True)

    if os.path.exists(zip_path):
        print(f"   📂 Unzipping {name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
    else:
        print(f"   ❌ ERROR: {zip_path} not found!")

print("\n✅ Data Ready! Local folders created.")

In [ ]:
import os
from google.colab import drive

# Check for the zips
base = "/content/drive/MyDrive/symAD-ECNN/data"
zips = [f"{base}/train_fast.zip", f"{base}/val_fast.zip", f"{base}/test_fast.zip"]

missing = [f for f in zips if not os.path.exists(f)]

if len(missing) == 0:
    print("✅ GOOD NEWS: Zip files found! Proceeding to extraction...")
else:
    print("⚠️ WARNING: Zip files missing. Please run the CNN-AE notebook first to create them.")
    print(f"   Missing: {missing}")

## ⚡ Turbo Data Loading (Local Disk)

**Why this matters**: Loading 33k+ files from Google Drive is SLOW (~30 min). Instead:
1. **Check** if zip files exist (created once)
2. **Extract** to local runtime disk (~2 min)
3. **Train** with blazing fast I/O (10-20x speedup)

**Note**: The zip files were created during CNN-AE training.

In [ ]:
# Keep Colab session alive
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button");
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click();
   }
   btn = document.querySelector('#ok');
   if (btn != null){
     console.log("Click connect button");
     btn.click();
   }
 }
 setInterval(ClickConnect, 60000)
'''))

print("✅ Keep-alive script activated!")

In [ ]:
# Import libraries (including e2cnn!)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pytorch_msssim import MS_SSIM

# E(2)-Equivariant CNN library
from e2cnn import gspaces
from e2cnn import nn as e2nn

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc
from glob import glob
import os
import time
import json
from tqdm import tqdm
import torchvision.transforms.functional as TF

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Device: {device}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Paths configuration
BASE_PATH = "/content/drive/MyDrive/symAD-ECNN"

# ⚡ DATA PATHS (Point to LOCAL DISK for speed) ⚡
IXI_TRAIN_PATH = "/content/local_data/train"
IXI_VAL_PATH   = "/content/local_data/val"
BRATS_PATH     = "/content/local_data/test"

# Model and results paths (Keep on Drive!)
MODEL_PATH = f"{BASE_PATH}/models/saved_models/ecnn_ae"
RESULTS_PATH = f"{BASE_PATH}/results/ecnn_autoencoder"

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("✅ All libraries imported (including e2cnn)!")
print(f"📁 Paths configured:")
print(f"   ⚡ Data (Local): {IXI_TRAIN_PATH}")
print(f"   💾 Models (Drive): {MODEL_PATH}")
print(f"   📊 Results (Drive): {RESULTS_PATH}")

## 2️⃣ Data Loading (Same as previous models)

In [ ]:
# Same data loading as previous notebooks

class MRIDataset(Dataset):
    """Dataset for loading preprocessed MRI slices (.npy files)"""
    def __init__(self, file_list):
        self.files = file_list
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        try:
            img = np.load(self.files[idx])
            img = np.expand_dims(img, 0)  # Add channel dimension
            return torch.from_numpy(img).float(), torch.from_numpy(img).float()
        except Exception as e:
            # Fallback for corrupted files
            print(f"Error loading {self.files[idx]}: {e}")
            return torch.zeros((1, 128, 128)), torch.zeros((1, 128, 128))

# Load file paths from LOCAL DISK
train_files = sorted(glob(f"{IXI_TRAIN_PATH}/*.npy"))
val_files = sorted(glob(f"{IXI_VAL_PATH}/*.npy"))
brats_files = sorted(glob(f"{BRATS_PATH}/*.npy"))

# Verify data exists
if len(train_files) == 0:
    raise ValueError(f"❌ No files found in {IXI_TRAIN_PATH}. Did you run the Turbo Loader?")

print(f"📊 Data Loaded from Local Disk:")
print(f"   Train: {len(train_files):,} slices")
print(f"   Val:   {len(val_files):,} slices")
print(f"   Test:  {len(brats_files):,} slices")

# Create DataLoaders
BATCH_SIZE = 64  # Increased for faster training (ECNN can handle it)

train_loader = DataLoader(
    MRIDataset(train_files), 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True
)
val_loader = DataLoader(
    MRIDataset(val_files), 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)
test_loader = DataLoader(
    MRIDataset(brats_files), 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

print(f"✅ DataLoaders created!")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

## 3️⃣ E(n)-Equivariant CNN-Autoencoder Architecture

### 🔑 Key Concepts:
- **Group**: C4 (4 discrete rotations)
- **Field Types**: Trivial (input) → Regular (features) → Trivial (output)
- **R2Conv**: Rotation-equivariant convolution
- **GeometricTensor**: Wrapper for equivariant operations

In [ ]:
class ECNNAutoencoder(nn.Module):
    """
    E(2)-Equivariant CNN Autoencoder
    
    Uses C4 group (4 rotations: 0°, 90°, 180°, 270°)
    with R2Conv layers for rotation equivariance
    """
    
    def __init__(self, latent_dim=256):
        super(ECNNAutoencoder, self).__init__()
        
        # Define E(2) group (C4 discrete rotations)
        self.r2_act = gspaces.Rot2dOnR2(N=4)  # C4 group
        
        # Input: trivial representation (scalar field - grayscale image)
        self.in_type = e2nn.FieldType(self.r2_act, [self.r2_act.trivial_repr])
        
        # Feature types (regular representation)
        self.feat_type_32 = e2nn.FieldType(self.r2_act, 32*[self.r2_act.regular_repr])
        self.feat_type_64 = e2nn.FieldType(self.r2_act, 64*[self.r2_act.regular_repr])
        self.feat_type_128 = e2nn.FieldType(self.r2_act, 128*[self.r2_act.regular_repr])
        self.feat_type_256 = e2nn.FieldType(self.r2_act, 256*[self.r2_act.regular_repr])
        
        # Equivariant Encoder
        self.encoder = nn.Sequential(
            # 128×128 -> 64×64
            e2nn.R2Conv(self.in_type, self.feat_type_32, kernel_size=3, padding=1),
            e2nn.IIDBatchNorm2d(self.feat_type_32),
            e2nn.ReLU(self.feat_type_32),
            e2nn.PointwiseMaxPool(self.feat_type_32, 2),
            
            # 64×64 -> 32×32
            e2nn.R2Conv(self.feat_type_32, self.feat_type_64, kernel_size=3, padding=1),
            e2nn.IIDBatchNorm2d(self.feat_type_64),
            e2nn.ReLU(self.feat_type_64),
            e2nn.PointwiseMaxPool(self.feat_type_64, 2),
            
            # 32×32 -> 16×16
            e2nn.R2Conv(self.feat_type_64, self.feat_type_128, kernel_size=3, padding=1),
            e2nn.IIDBatchNorm2d(self.feat_type_128),
            e2nn.ReLU(self.feat_type_128),
            e2nn.PointwiseMaxPool(self.feat_type_128, 2),
            
            # 16×16 -> 8×8
            e2nn.R2Conv(self.feat_type_128, self.feat_type_256, kernel_size=3, padding=1),
            e2nn.IIDBatchNorm2d(self.feat_type_256),
            e2nn.ReLU(self.feat_type_256),
            e2nn.PointwiseMaxPool(self.feat_type_256, 2)
        )
        
        # Group pooling (make rotation-invariant)
        self.group_pool = e2nn.GroupPooling(self.feat_type_256)
        
        # Latent space (non-equivariant)
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)
        
        # Decoder (equivariant upsampling)
        self.upconv1 = e2nn.R2Conv(self.feat_type_256, self.feat_type_256, kernel_size=3, padding=1)
        self.bn1 = e2nn.IIDBatchNorm2d(self.feat_type_256)
        self.relu1 = e2nn.ReLU(self.feat_type_256)
        
        self.upconv2 = e2nn.R2Conv(self.feat_type_256, self.feat_type_128, kernel_size=3, padding=1)
        self.bn2 = e2nn.IIDBatchNorm2d(self.feat_type_128)
        self.relu2 = e2nn.ReLU(self.feat_type_128)
        
        self.upconv3 = e2nn.R2Conv(self.feat_type_128, self.feat_type_64, kernel_size=3, padding=1)
        self.bn3 = e2nn.IIDBatchNorm2d(self.feat_type_64)
        self.relu3 = e2nn.ReLU(self.feat_type_64)
        
        self.upconv4 = e2nn.R2Conv(self.feat_type_64, self.feat_type_32, kernel_size=3, padding=1)
        self.bn4 = e2nn.IIDBatchNorm2d(self.feat_type_32)
        self.relu4 = e2nn.ReLU(self.feat_type_32)
        
        # Final layer (back to trivial representation)
        self.final_conv = e2nn.R2Conv(self.feat_type_32, self.in_type, kernel_size=3, padding=1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Wrap as GeometricTensor
        x_g = e2nn.GeometricTensor(x, self.in_type)
        
        # Equivariant encoding
        features = self.encoder(x_g)
        
        # Group pooling (rotation-invariant)
        invariant = self.group_pool(features)
        
        # Latent space
        batch_size = invariant.size(0)
        flat = invariant.tensor.view(batch_size, -1)
        z = self.fc_encode(flat)
        
        # Decode
        decoded_flat = self.fc_decode(z)
        decoded_features = decoded_flat.view(batch_size, 256, 8, 8)
        decoded_g = e2nn.GeometricTensor(decoded_features, self.feat_type_256)
        
        # Equivariant decoder with upsampling
        x = self.upconv1(decoded_g)
        x = self.bn1(x)
        x = self.relu1(x)
        x = e2nn.GeometricTensor(nn.functional.interpolate(x.tensor, scale_factor=2, mode='bilinear'), x.type)
        
        x = self.upconv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = e2nn.GeometricTensor(nn.functional.interpolate(x.tensor, scale_factor=2, mode='bilinear'), x.type)
        
        x = self.upconv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = e2nn.GeometricTensor(nn.functional.interpolate(x.tensor, scale_factor=2, mode='bilinear'), x.type)
        
        x = self.upconv4(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = e2nn.GeometricTensor(nn.functional.interpolate(x.tensor, scale_factor=2, mode='bilinear'), x.type)
        
        # Final output
        x = self.final_conv(x)
        x_recon = self.sigmoid(x.tensor)
        
        return x_recon
    
    def get_latent(self, x):
        x_g = e2nn.GeometricTensor(x, self.in_type)
        features = self.encoder(x_g)
        invariant = self.group_pool(features)
        flat = invariant.tensor.view(invariant.size(0), -1)
        return self.fc_encode(flat)

model = ECNNAutoencoder().to(device)
total_params = sum(p.numel() for p in model.parameters())

print("⚡ E(n)-Equivariant CNN-Autoencoder Created!")
print(f"   Total parameters: {total_params:,}")
print(f"   Group: C4 (4-fold rotational symmetry)")
print(f"   Equivariance: Rotations + Translations")

## 4️⃣ Test Rotation Equivariance Property

In [ ]:
# Test equivariance: f(rotate(x)) should ≈ rotate(f(x))

model.eval()
sample = next(iter(val_loader))[0][:1].to(device)  # One sample

with torch.no_grad():
    # Original reconstruction
    recon_original = model(sample)
    
    # Rotate input 90°, then reconstruct
    sample_rot90 = TF.rotate(sample, 90)
    recon_rot90 = model(sample_rot90)
    
    # Reconstruct original, then rotate 90°
    recon_then_rot90 = TF.rotate(recon_original, 90)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

axes[0, 0].imshow(sample[0, 0].cpu(), cmap='gray')
axes[0, 0].set_title('Original Input')
axes[0, 0].axis('off')

axes[0, 1].imshow(sample_rot90[0, 0].cpu(), cmap='gray')
axes[0, 1].set_title('Rotated Input (90°)')
axes[0, 1].axis('off')

axes[0, 2].imshow(recon_original[0, 0].cpu(), cmap='gray')
axes[0, 2].set_title('Recon(Original)')
axes[0, 2].axis('off')

axes[1, 0].imshow(recon_rot90[0, 0].cpu(), cmap='gray')
axes[1, 0].set_title('Recon(Rotated) - Method A')
axes[1, 0].axis('off')

axes[1, 1].imshow(recon_then_rot90[0, 0].cpu(), cmap='gray')
axes[1, 1].set_title('Rotate(Recon) - Method B')
axes[1, 1].axis('off')

difference = torch.abs(recon_rot90 - recon_then_rot90)
axes[1, 2].imshow(difference[0, 0].cpu(), cmap='hot')
axes[1, 2].set_title(f'Difference (MSE: {difference.mean():.6f})')
axes[1, 2].axis('off')

plt.suptitle('Rotation Equivariance Test: Should be nearly identical', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/ecnn_equivariance_test.png')
plt.show()

print("✅ Equivariance test complete!")
print(f"   MSE difference: {difference.mean():.8f} (lower is better)")
print("   If MSE < 0.01, model is approximately equivariant!")

## 5️⃣ Training (Same as previous models)

In [ ]:
# Training setup with mixed precision and better loss

from torch.cuda.amp import autocast, GradScaler

class CombinedLoss(nn.Module):
    """Combined MSE + SSIM Loss for better perceptual quality"""
    def __init__(self, alpha=0.84):
        super().__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.ssim = MS_SSIM(data_range=1.0, size_average=True, channel=1)
    
    def forward(self, output, target):
        mse_loss = self.mse(output, target)
        ssim_loss = 1 - self.ssim(output, target)
        return self.alpha * mse_loss + (1 - self.alpha) * ssim_loss

criterion = CombinedLoss(alpha=0.84)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
scaler = GradScaler()  # Mixed precision training

def train_epoch(model, dataloader, criterion, optimizer, device, scaler):
    """Train with mixed precision for 2-3x speedup"""
    model.train()
    running_loss = 0.0
    pbar = tqdm(dataloader, desc='Training')
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with autocast():
            output = model(data)
            loss = criterion(output, target)
        
        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    return running_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            running_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    return running_loss / len(dataloader)

print("✅ Training setup complete with mixed precision!")

In [ ]:
# Main training loop with checkpointing
NUM_EPOCHS = 50  # 50 epochs for good convergence
KEEP_LAST_N_CHECKPOINTS = 3

# Check for existing checkpoints
checkpoints = sorted(glob(f'{MODEL_PATH}/ecnn_epoch*.pth'))
RESUME = len(checkpoints) > 0

if RESUME:
    latest = checkpoints[-1]
    print(f"✅ Found checkpoint: {os.path.basename(latest)}")
    checkpoint = torch.load(latest, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch']
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))
    best_epoch = checkpoint.get('best_epoch', 0)
    print(f"   Resuming from epoch {start_epoch}")
    print(f"   Current LR: {optimizer.param_groups[0]['lr']:.2e}")
else:
    start_epoch = 0
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_epoch = 0
    print("📝 Starting fresh ECNN training")

print(f"\n🚀 Training E(n)-Equivariant CNN-AE: Epochs {start_epoch + 1} to {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}, Device: {device}")
print(f"   Mixed precision: Enabled")
print("-" * 60)

start_time = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()
    
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:3d}/{NUM_EPOCHS}] | Train: {train_loss:.6f} | Val: {val_loss:.6f} | LR: {current_lr:.2e} | Time: {epoch_time/60:.1f}min")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'best_val_loss': best_val_loss,
            'best_epoch': best_epoch
        }, f'{MODEL_PATH}/ecnn_best.pth')
        print(f"   ✅ Best ECNN model saved!")
    
    # Save checkpoint every epoch
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch
    }, f'{MODEL_PATH}/ecnn_epoch{epoch+1}.pth')
    
    # Cleanup old checkpoints
    checkpoints = sorted(glob(f'{MODEL_PATH}/ecnn_epoch*.pth'))
    if len(checkpoints) > KEEP_LAST_N_CHECKPOINTS:
        for old_ckpt in checkpoints[:-KEEP_LAST_N_CHECKPOINTS]:
            os.remove(old_ckpt)
            print(f"   🗑️ Deleted old checkpoint: {os.path.basename(old_ckpt)}")
    
    print("-" * 60)

total_time = time.time() - start_time
print(f"\n🎉 ECNN Training Complete!")
print(f"   Total Time: {total_time/3600:.2f} hours")
print(f"   Best Epoch: {best_epoch}, Best Val Loss: {best_val_loss:.6f}")

# Plot training curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train', linewidth=2)
plt.plot(val_losses, label='Val', linewidth=2)
plt.axvline(x=best_epoch-1, color='r', linestyle='--', label=f'Best ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ECNN-AE Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train', linewidth=2)
plt.plot(val_losses, label='Val', linewidth=2)
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss (log)')
plt.title('ECNN-AE Training (Log Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/ecnn_training_curves.png', dpi=150)
plt.show()

## 6️⃣ Evaluation and Comparison with Other Models

In [ ]:
# Comprehensive evaluation with metrics

# Load best model
checkpoint = torch.load(f'{MODEL_PATH}/ecnn_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Best ECNN model loaded (Epoch {checkpoint['epoch']}, Val Loss: {checkpoint['val_loss']:.6f})")

def calculate_reconstruction_error(model, dataloader, device):
    """Calculate per-sample reconstruction errors"""
    model.eval()
    errors = []
    with torch.no_grad():
        for data, _ in tqdm(dataloader, desc='Computing errors'):
            data = data.to(device)
            recon = model(data)
            mse = nn.functional.mse_loss(recon, data, reduction='none')
            mse = mse.view(mse.size(0), -1).mean(dim=1)
            errors.extend(mse.cpu().numpy())
    return np.array(errors)

normal_errors = calculate_reconstruction_error(model, val_loader, device)
anomaly_errors = calculate_reconstruction_error(model, test_loader, device)

# Calculate metrics
y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(anomaly_errors))])
y_scores = np.concatenate([normal_errors, anomaly_errors])

auroc = roc_auc_score(y_true, y_scores)
precision, recall, _ = precision_recall_curve(y_true, y_scores)
auprc = auc(recall, precision)

print(f"\n📈 E(n)-Equivariant CNN-Autoencoder Performance:")
print(f"   AUROC: {auroc:.4f} 🏆")
print(f"   AUPRC: {auprc:.4f}")
print(f"   Normal errors: {normal_errors.mean():.6f} ± {normal_errors.std():.6f}")
print(f"   Anomaly errors: {anomaly_errors.mean():.6f} ± {anomaly_errors.std():.6f}")

# Plot distributions and ROC
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(normal_errors, bins=50, alpha=0.7, label='Normal (Healthy)', density=True, color='blue')
plt.hist(anomaly_errors, bins=50, alpha=0.7, label='Anomaly (Tumor)', density=True, color='red')
plt.xlabel('Reconstruction Error', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend(fontsize=11)
plt.title('ECNN-AE: Error Distribution', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_true, y_scores)
plt.plot(fpr, tpr, label=f'ECNN-AE (AUROC={auroc:.4f})', linewidth=2, color='green')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(fontsize=11)
plt.title('ECNN-AE: ROC Curve', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/ecnn_evaluation.png', dpi=150)
plt.show()

# Save results
results = {
    'model': 'ECNN-Autoencoder',
    'auroc': float(auroc),
    'auprc': float(auprc),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'normal_error_mean': float(normal_errors.mean()),
    'anomaly_error_mean': float(anomaly_errors.mean()),
    'total_params': total_params,
    'training_time_hours': total_time / 3600
}

with open(f'{RESULTS_PATH}/ecnn_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("\n✅ E(n)-Equivariant CNN-Autoencoder evaluation complete!")

## 7️⃣ Visualization: Reconstruction Maps

Visualize how the ECNN reconstructs normal and anomalous samples.

In [ ]:
def visualize_reconstructions(model, dataloader, device, num_samples=5, title_prefix=""):
    """Visualize original images, reconstructions, and error maps"""
    model.eval()
    
    # Get samples
    data, _ = next(iter(dataloader))
    data = data[:num_samples].to(device)
    
    # Generate reconstructions
    with torch.no_grad():
        recon = model(data)
    
    # Calculate error maps
    error_maps = torch.abs(data - recon)
    
    # Move to CPU
    data_np = data.cpu().numpy()
    recon_np = recon.cpu().numpy()
    error_np = error_maps.cpu().numpy()
    
    # Create figure
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        # Original
        axes[i, 0].imshow(data_np[i, 0], cmap='gray', vmin=0, vmax=1)
        axes[i, 0].set_title(f'{title_prefix} Original', fontsize=12, fontweight='bold')
        axes[i, 0].axis('off')
        
        # Reconstruction
        axes[i, 1].imshow(recon_np[i, 0], cmap='gray', vmin=0, vmax=1)
        mse_val = np.mean((data_np[i, 0] - recon_np[i, 0])**2)
        axes[i, 1].set_title(f'Reconstruction (MSE={mse_val:.6f})', fontsize=12, fontweight='bold')
        axes[i, 1].axis('off')
        
        # Error map
        im = axes[i, 2].imshow(error_np[i, 0], cmap='hot', vmin=0, vmax=error_np[i, 0].max())
        axes[i, 2].set_title(f'Error Map (Max={error_np[i, 0].max():.4f})', fontsize=12, fontweight='bold')
        axes[i, 2].axis('off')
        plt.colorbar(im, ax=axes[i, 2], fraction=0.046)
    
    plt.tight_layout()
    return fig

# Visualize Normal Cases
print("🔍 Visualizing Normal Cases (Healthy Brains)...")
fig_normal = visualize_reconstructions(model, val_loader, device, num_samples=5, title_prefix="Normal")
plt.savefig(f'{RESULTS_PATH}/ecnn_reconstruction_normal.png', dpi=150, bbox_inches='tight')
plt.show()

# Visualize Anomaly Cases
print("🔍 Visualizing Anomaly Cases (Brain Tumors)...")
fig_anomaly = visualize_reconstructions(model, test_loader, device, num_samples=5, title_prefix="Anomaly")
plt.savefig(f'{RESULTS_PATH}/ecnn_reconstruction_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("📊 INTERPRETATION:")
print("="*60)
print("🔵 NORMAL: Low MSE, minimal red in error maps")
print("   → ECNN learned healthy brain patterns well")
print("🔴 ANOMALY: High MSE, bright red regions")
print("   → Tumor regions have high reconstruction error")
print("   → Thanks to rotation equivariance, errors are consistent")
print("="*60)

In [ ]:
# Compare all six models (load results from JSON files)

import json

# Load all results
BASE_RESULTS_PATH = "/content/drive/MyDrive/symAD-ECNN/results"

with open(f'{BASE_RESULTS_PATH}/cnn_autoencoder/cnn_results.json', 'r') as f:
    cnn_baseline_results = json.load(f)

with open(f'{BASE_RESULTS_PATH}/cnn_ae_augmented/cnn_results.json', 'r') as f:
    cnn_aug_results = json.load(f)

with open(f'{BASE_RESULTS_PATH}/resnet_feature_distance/resnet_results.json', 'r') as f:
    resnet_feat_results = json.load(f)

with open(f'{BASE_RESULTS_PATH}/resnet_autoencoder/resnet_results.json', 'r') as f:
    resnet_ae_results = json.load(f)

with open(f'{BASE_RESULTS_PATH}/resnet_finetuned/resnet_results.json', 'r') as f:
    resnet_ft_results = json.load(f)

with open(f'{RESULTS_PATH}/ecnn_results.json', 'r') as f:
    ecnn_results = json.load(f)

# Create comparison table
print("\n" + "="*90)
print("🏆 FINAL COMPARISON - ALL SIX MODELS")
print("="*90)
print(f"{'Model':<40} {'AUROC':<10} {'AUPRC':<10} {'Val Loss':<10}")
print("-"*90)
print(f"{'02 - CNN-AE Baseline':<40} {cnn_baseline_results['auroc']:<10.4f} {cnn_baseline_results['auprc']:<10.4f} {cnn_baseline_results['best_val_loss']:<10.6f}")
print(f"{'03 - CNN-AE + Augmentation':<40} {cnn_aug_results['auroc']:<10.4f} {cnn_aug_results['auprc']:<10.4f} {cnn_aug_results['best_val_loss']:<10.6f}")
print(f"{'04 - ResNet Features + Distance':<40} {resnet_feat_results['auroc']:<10.4f} {resnet_feat_results['auprc']:<10.4f} {resnet_feat_results.get('best_val_loss', 0.0):<10.6f}")
print(f"{'05 - ResNet Encoder (Frozen)':<40} {resnet_ae_results['auroc']:<10.4f} {resnet_ae_results['auprc']:<10.4f} {resnet_ae_results['best_val_loss']:<10.6f}")
print(f"{'06 - ResNet Fine-tuned':<40} {resnet_ft_results['auroc']:<10.4f} {resnet_ft_results['auprc']:<10.4f} {resnet_ft_results['best_val_loss']:<10.6f}")
print(f"{'07 - ECNN (Main Contribution) ⭐':<40} {ecnn_results['auroc']:<10.4f} {ecnn_results['auprc']:<10.4f} {ecnn_results['best_val_loss']:<10.6f}")
print("="*90)

# Plot comparison
models = ['CNN-AE\nBaseline', 'CNN-AE\n+Aug', 'ResNet\nFeatures', 'ResNet\nFrozen', 'ResNet\nFine-tuned', 'ECNN\n(Ours)']
aurocs = [
    cnn_baseline_results['auroc'], 
    cnn_aug_results['auroc'], 
    resnet_feat_results['auroc'],
    resnet_ae_results['auroc'],
    resnet_ft_results['auroc'],
    ecnn_results['auroc']
]
auprcs = [
    cnn_baseline_results['auprc'], 
    cnn_aug_results['auprc'], 
    resnet_feat_results['auprc'],
    resnet_ae_results['auprc'],
    resnet_ft_results['auprc'],
    ecnn_results['auprc']
]

colors = ['#3498db', '#9b59b6', '#f39c12', '#e67e22', '#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(models, aurocs, color=colors, alpha=0.8)
axes[0].set_ylabel('AUROC', fontsize=12)
axes[0].set_title('Model Comparison - AUROC', fontsize=14, fontweight='bold')
axes[0].set_ylim([0.6, 1.0])
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(aurocs):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

axes[1].bar(models, auprcs, color=colors, alpha=0.8)
axes[1].set_ylabel('AUPRC', fontsize=12)
axes[1].set_title('Model Comparison - AUPRC', fontsize=14, fontweight='bold')
axes[1].set_ylim([0.6, 1.0])
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(auprcs):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ All results saved in: {RESULTS_PATH}/")
print("🎉 Project complete! You now have six trained models for comprehensive comparison.")